In [ ]:
# import sys
# from pathlib import Path

# # 将 mas_2 根目录加入路径（内含 src/）。cwd 为 notebooks/ 或 mas_2/ 均可。
# _cwd = Path().resolve()
# for _root in (_cwd, _cwd.parent):
#     if (_root / "src").is_dir():
#         sys.path.insert(0, str(_root))
#         break

In [ ]:
import os
import sys
import time
import datetime

class TeeLogger:
    def __init__(self, filename, terminal):
        self.terminal = terminal
        log_dir = os.path.dirname(filename)
        if log_dir:
            os.makedirs(log_dir, exist_ok=True)
        self.log = open(filename, "a", encoding="utf-8")

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()

    def flush(self):
        self.terminal.flush()
        self.log.flush()


def log_event(message, now=None):
    now = now or datetime.datetime.now()
    print(f"[{now.strftime('%Y-%m-%d %H:%M:%S')}] {message}", flush=True)


def run_graph_with_timing(graph, initial_state, config):
    total_start_wall = datetime.datetime.now()
    total_start_perf = time.perf_counter()
    step_start_wall = total_start_wall
    step_start_perf = total_start_perf
    final_state = None
    step_timings = []

    try:
        for step in graph.stream(initial_state, config=config):
            for node_name, state in step.items():
                step_end_wall = datetime.datetime.now()
                step_end_perf = time.perf_counter()
                elapsed = step_end_perf - step_start_perf

                log_event(f"START node={node_name}", now=step_start_wall)
                log_event(f"END node={node_name} elapsed={elapsed:.2f}s", now=step_end_wall)

                step_timings.append({
                    "node": node_name,
                    "started_at": step_start_wall.strftime("%Y-%m-%d %H:%M:%S"),
                    "ended_at": step_end_wall.strftime("%Y-%m-%d %H:%M:%S"),
                    "elapsed_seconds": round(elapsed, 2),
                })
                final_state = state
                step_start_wall = step_end_wall
                step_start_perf = step_end_perf
    finally:
        total_elapsed = time.perf_counter() - total_start_perf
        log_event(f"RUN COMPLETE total_elapsed={total_elapsed:.2f}s")

    return final_state, step_timings


# 同步输出到终端和文件
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
log_filename = f"logs/run_log_{timestamp}.log"
original_stdout = sys.stdout
original_stderr = sys.stderr
logger = TeeLogger(log_filename, original_stdout)
sys.stdout = logger
sys.stderr = logger
log_event(f"=== 开始运行，日志将同步保存至 {log_filename} ===")

try:
    from src.main import graph
    from langchain_core.messages import HumanMessage

    initial_state = {
        "messages": [HumanMessage(content="我想用data目录下的bmmc_b_cell.h5ad单细胞数据进行分析")],
        "user_query": "我想用data目录下的bmmc_b_cell.h5ad单细胞数据进行分析",
        "plan": [],
        "current_step_index": 0,
        "next_worker": None,
        "last_worker": None,
        "pending_contribution": None,
        "critique_feedback": None,
        "is_approved": False,
    }

    config = {"configurable": {"thread_id": "test-001"}}
    final_state, step_timings = run_graph_with_timing(graph, initial_state, config)
    print("【最终结果】\n", final_state["final_answer"])
    print("【步骤耗时】")
    for item in step_timings:
        print(
            f"- {item['node']}: {item['elapsed_seconds']:.2f}s "
            f"({item['started_at']} -> {item['ended_at']})"
        )
finally:
    sys.stdout = original_stdout
    sys.stderr = original_stderr
    logger.log.close()